In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import joblib
ticker = "BBRI.JK"
data = yf.download(ticker, start="2021-01-01", end="2026-01-01")
prices = data['Close'].values.astype(float)
split_idx = int(len(prices) * 0.8)
train_prices = prices[:split_idx].reshape(-1, 1)
test_prices = prices[split_idx:].reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_prices).flatten()
test_scaled = scaler.transform(test_prices).flatten()

def create_sequences(data_array, seq_len):
    X, y = [], []
    for i in range(len(data_array) - seq_len):
        X.append(data_array[i : i + seq_len])
        y.append(data_array[i + seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 10

X_train, y_train = create_sequences(train_scaled, SEQ_LEN)
X_test, y_test = create_sequences(test_scaled, SEQ_LEN)

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

/tmp/ipykernel_1080/1381782604.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2021-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


In [ ]:

class StockLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2):
        super(StockLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StockLSTM().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
batch_size = 16

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for i in range(0, len(X_train_t), batch_size):
        xb = X_train_t[i:i+batch_size].to(device)
        yb = y_train_t[i:i+batch_size].to(device)

        optimizer.zero_grad()
        predictions = model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(xb)

    total_loss = epoch_loss / len(X_train_t)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.6f}")

Epoch 10/50 - Loss: 0.006241
Epoch 20/50 - Loss: 0.004132
Epoch 30/50 - Loss: 0.003337
Epoch 40/50 - Loss: 0.002922
Epoch 50/50 - Loss: 0.002481


In [ ]:

model.eval()
with torch.no_grad():

    test_preds_scaled = model(X_test_t.to(device)).cpu().numpy()

test_preds = scaler.inverse_transform(test_preds_scaled.reshape(-1, 1)).flatten()
actuals = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
mae = np.mean(np.abs(test_preds - actuals))
rmse = np.sqrt(np.mean((test_preds - actuals) ** 2))
mape = np.mean(np.abs((actuals - test_preds) / actuals)) * 100

print("\n=== HASIL EVALUASI MODEL ===")
print(f"Mean Absolute Error (MAE): Rp {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): Rp {rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")


=== HASIL EVALUASI MODEL ===
Mean Absolute Error (MAE): Rp 122.37
Root Mean Squared Error (RMSE): Rp 143.40
Mean Absolute Percentage Error (MAPE): 3.60%


In [ ]:

torch.save(model.state_dict(), 'lstm_stock_model.pth')
print("✅ Bobot model berhasil disimpan ke 'lstm_stock_model.pth'")

joblib.dump(scaler, 'scaler.pkl')
print("✅ Scaler berhasil disimpan ke 'scaler.pkl'")

✅ Bobot model berhasil disimpan ke 'lstm_stock_model.pth'
✅ Scaler berhasil disimpan ke 'scaler.pkl'
